# Exploração da API do Tesouro Nacional

Objetivo: identificar os endpoints e o formato dos dados de:
- Resultado primário do Governo Central
- Gasto público (despesa)

**Fontes candidatas:**
- API Tesouro Transparente: https://apidatalake.tesouro.gov.br/
- CKAN: https://www.tesourotransparente.gov.br/ckan/api/3/action/datastore_search

In [ ]:
import requests
import json

# Testar endpoint do Resultado Primário
URL_RESULTADO = "https://apidatalake.tesouro.gov.br/ords/siconfi/tt/resultado_primario"

r = requests.get(URL_RESULTADO, timeout=15, params={"an_exercicio": 2026})
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()

# Ver o começo do conteúdo
print(r.text[:1000])

In [ ]:
# Alternativa 1: Documentação da API
URL_DOC = "https://apidatalake.tesouro.gov.br/docs/"
r = requests.get(URL_DOC, timeout=15)
print(f"Docs: {r.status_code}")

# Alternativa 2: CKAN (dados abertos) - buscar por "resultado primário"
URL_CKAN = "https://www.tesourotransparente.gov.br/ckan/api/3/action/package_search"
r = requests.get(URL_CKAN, params={"q": "resultado primário"}, timeout=15)
print(f"CKAN busca: {r.status_code}")

# Se CKAN retornar 200, ver os primeiros resultados
if r.status_code == 200:
    dados = r.json()
    print(f"Total encontrados: {dados['result']['count']}")
    for pkg in dados['result']['results'][:5]:
        print(f"  - {pkg['title']}")

In [ ]:
# Teste A: Grandes Números — Resultado Primário do Governo Central
URL_GN = "https://grandesnumeros.tesouro.gov.br/resultado_primario"

r = requests.get(URL_GN, timeout=15)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()
print(r.text[:2000])

In [ ]:
# Detalhar o dataset "Resultado do Tesouro Nacional - Série Histórica"
URL_CKAN_DATASET = "https://www.tesourotransparente.gov.br/ckan/api/3/action/package_show"
r = requests.get(URL_CKAN_DATASET, params={"id": "resultado-do-tesouro-nacional-serie-historica"}, timeout=15)

if r.status_code == 200:
    dados = r.json()
    pkg = dados["result"]
    print(f"Título: {pkg['title']}")
    print(f"Notas: {pkg.get('notes', '')[:200]}")
    print(f"\nRecursos disponíveis:")
    for recurso in pkg["resources"]:
        print(f"  - {recurso['name']}")
        print(f"    Formato: {recurso['format']}")
        print(f"    URL: {recurso['url']}")
        print()
else:
    print(f"Status: {r.status_code}")
    print(r.text[:500])

In [ ]:
URL_CKAN_DATASET = "https://www.tesourotransparente.gov.br/ckan/api/3/action/package_show"
r = requests.get(URL_CKAN_DATASET, params={"id": "despesas-da-uniao-mensais-desde-2008"}, timeout=15)

if r.status_code == 200:
    dados = r.json()
    pkg = dados["result"]
    print(f"Título: {pkg['title']}")
    print(f"\nRecursos disponíveis:")
    for recurso in pkg["resources"][:10]:  # primeiros 10
        print(f"  - {recurso['name']} ({recurso['format']})")
        print(f"    {recurso['url']}")
else:
    print(f"Status: {r.status_code}")

In [ ]:
URL_CKAN = "https://www.tesourotransparente.gov.br/ckan/api/3/action/package_search"
r = requests.get(URL_CKAN, params={"q": "resultado primário"}, timeout=15)
dados = r.json()
for pkg in dados['result']['results']:
    print(f"Nome (slug): {pkg['name']}")
    print(f"Título: {pkg['title']}")
    print()

In [ ]:
URL_CKAN_DATASET = "https://www.tesourotransparente.gov.br/ckan/api/3/action/package_show"
r = requests.get(URL_CKAN_DATASET, params={"id": "resultado-do-tesouro-nacional"}, timeout=15)

if r.status_code == 200:
    dados = r.json()
    pkg = dados["result"]
    print(f"Título: {pkg['title']}")
    print(f"Notas: {pkg.get('notes', '')[:300]}")
    print(f"\nRecursos disponíveis:")
    for recurso in pkg["resources"]:
        print(f"  - {recurso['name']}")
        print(f"    Formato: {recurso['format']}")
        print(f"    URL: {recurso['url']}")
        print()
else:
    print(f"Status: {r.status_code}")
    print(r.text[:500])

In [ ]:
# Testar séries candidatas do BCB para finanças públicas
SERIES_CANDIDATAS = {
    4649: "Resultado primário do Governo Central (% PIB) - candidato 1",
    5727: "Gasto do Governo Central (% PIB) - candidato 1",
    5793: "Resultado primário do Governo Central - candidato 2",
}

for codigo, descricao in SERIES_CANDIDATAS.items():
    try:
        url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados/ultimos/2?formato=json"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            dados = r.json()
            print(f"✅ {codigo} — {descricao}")
            for d in dados:
                print(f"    {d['data']} → {d['valor']}")
        else:
            print(f"❌ {codigo} — status {r.status_code}")
    except Exception as e:
        print(f"❌ {codigo} — erro: {e}")
    print()

In [ ]:
# Explorar a API de Séries Temporais do Tesouro
URL_API_TESOURO = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/docs"

r = requests.get(URL_API_TESOURO, timeout=15)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()
print(r.text[:3000])

In [ ]:
# Listar todas as séries disponíveis
URL_SERIES = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/series"

r = requests.get(URL_SERIES, timeout=15)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get('Content-Type')}")
print()

dados = r.json()
print(f"Status da API: {dados.get('status')}")
print(f"Total de séries: {len(dados.get('registros', []))}")
print()

# Mostrar as primeiras 30 séries
for serie in dados.get("registros", [])[:30]:
    # Pega as duas primeiras chaves do dict (assumindo código e nome)
    chaves = list(serie.keys())
    codigo = serie.get(chaves[0], "?")
    nome = serie.get(chaves[1], "?")
    print(f"  {codigo} — {nome}")

In [ ]:
# Testar o endpoint de dados para o tema 10
URL_DADOS = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/resultado-fiscal"

params = {
    "tema": 10,
    # Não passamos codigo_da_serie → retorna todas as séries do tema
}

r = requests.get(URL_DADOS, params=params, timeout=15)
print(f"Status: {r.status_code}")
print()

dados = r.json()
print(f"Status da API: {dados.get('status')}")
print(f"Total de registros: {len(dados.get('registros', []))}")
print()

# Mostrar os primeiros 5 registros
for reg in dados.get("registros", [])[:5]:
    print(f"  {reg['data']} | {reg['nomeSerie']} | {reg['valor']}")

In [ ]:
# Ver a estrutura real dos registros de séries
URL_SERIES = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/series"

r = requests.get(URL_SERIES, timeout=15)
dados = r.json()

print(f"Status: {dados.get('status')}")
print(f"Total: {len(dados.get('registros', []))}")
print()

# Ver o PRIMEIRO registro completo
if dados.get("registros"):
    primeiro = dados["registros"][0]
    print("Estrutura do primeiro registro:")
    for chave, valor in primeiro.items():
        print(f"  {chave!r}: {valor!r}")

In [ ]:
# Ver quais NOMES de série existem no tema 10
nomes_unicos = set()
for reg in dados["registros"]:
    nomes_unicos.add(reg["nomeSerie"])

print(f"Total de séries distintas no tema 10: {len(nomes_unicos)}")
print()
for nome in sorted(nomes_unicos):
    print(f"  - {nome}")

In [ ]:
URL_SERIES = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/series"
r = requests.get(URL_SERIES, timeout=15)
dados = r.json()

if dados.get("registros"):
    primeiro = dados["registros"][0]
    print("Estrutura do primeiro registro:")
    for chave, valor in primeiro.items():
        print(f"  {chave!r}: {valor!r}")

In [ ]:
URL_DADOS = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/resultado-fiscal"
r = requests.get(URL_DADOS, params={"tema": 10}, timeout=15)
dados = r.json()

nomes = set(reg["nomeSerie"] for reg in dados["registros"])
print(f"Séries distintas no tema 10: {len(nomes)}")
for nome in sorted(nomes):
    print(f"  - {nome}")

In [ ]:
# Listar TODAS as séries, agrupadas por subtema
URL_SERIES = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/series"

r = requests.get(URL_SERIES, timeout=15)
dados = r.json()
registros = dados["registros"]

print(f"Total de séries: {len(registros)}\n")

# Agrupar por subtema
from collections import defaultdict
por_subtema = defaultdict(list)

for reg in registros:
    chave = f"{reg['codigoTema']} — {reg['nomeSubtema']}"
    por_subtema[chave].append({
        "codigo": reg["codigoSerie"],
        "nome": reg["nomeSerie"],
    })

# Mostrar cada subtema e suas séries
for subtema, series in sorted(por_subtema.items()):
    print(f"\n=== {subtema} ({len(series)} séries) ===")
    for s in series:
        print(f"  {s['codigo']}  →  {s['nome']}")

In [ ]:
# Testar chamada filtrada para o Resultado Primário (10.04.1)
URL_DADOS = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/resultado-fiscal"

params = {
    "tema": 10,
    "codigo_da_serie": "10.04.1",
    "data_inicio": "06/2026",
    "data_fim": "07/2026",
}

r = requests.get(URL_DADOS, params=params, timeout=15)
print(f"Status: {r.status_code}\n")

dados = r.json()
print(f"Status: {dados.get('status')}")
print(f"Registros retornados: {len(dados.get('registros', []))}\n")

for reg in dados.get("registros", []):
    print(f"  {reg['data']} | {reg['nomeSerie']} | {reg['valor']}")

In [ ]:
# Testar chamada filtrada para Despesa Total (10.03.1)
params = {
    "tema": 10,
    "codigo_da_serie": "10.03.1",
    "data_inicio": "06/2026",
    "data_fim": "07/2026",
}

r = requests.get(URL_DADOS, params=params, timeout=15)
dados = r.json()
print(f"Status: {dados.get('status')}")
print(f"Registros: {len(dados.get('registros', []))}\n")

for reg in dados.get("registros", []):
    print(f"  {reg['data']} | {reg['nomeSerie']} | {reg['valor']}")

In [ ]:
# Debugar: o que a API retorna com data_inicio calculada?
from datetime import datetime, timedelta

# Mesma lógica do _meses_atras
hoje = datetime.today()
base = hoje.replace(day=1)
alvo = base - timedelta(days=13 * 30)
data_inicio = alvo.strftime("%m/%Y")
print(f"Hoje: {hoje.strftime('%d/%m/%Y')}")
print(f"data_inicio calculada: {data_inicio}")
print()

# Fazer a chamada com essa data_inicio
URL_DADOS = "https://apiapex.tesouro.gov.br/aria/v1/series-temporais/custom/resultado-fiscal"
params = {
    "tema": 10,
    "codigo_da_serie": "10.04.1",
    "data_inicio": data_inicio,
}
r = requests.get(URL_DADOS, params=params, timeout=15)
dados = r.json()

print(f"Total de registros: {len(dados.get('registros', []))}")
print()
print("Datas retornadas:")
for reg in dados.get("registros", []):
    print(f"  {reg['data']}")